In [0]:
from pyspark.sql.functions import *

In [0]:
# Get parameters from widgets
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
%sql
use catalog :catalog_name

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog_name}.23283_metadata")

Metadata Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.metadata_23283.pipeline_control (
  table_name STRING,
  last_run_timestamp TIMESTAMP
)
""")

### Helper Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from datetime import *

def get_last_run(table):
    df = spark.sql(f"""
        SELECT MAX(last_run_timestamp) as last_run 
        FROM {catalog_name}.metadata_23283.pipeline_control 
        WHERE table_name = '{table}'
    """)
    
    result = df.collect()[0]["last_run"]
    return result if result else datetime(1900,1,1)

### Incremental + Dedup

In [0]:
def process_table(table, primary_key):
    
    bronze_df = spark.table(f"{catalog_name}.bronze_23283.{table}")
    last_run = get_last_run(table)

    filtered_df = bronze_df.filter(
        (col("created_at") > last_run) |
        (col("updated_at") > last_run)
    )

    window = Window.partitionBy(primary_key).orderBy(col("updated_at").desc())

    dedup_df = filtered_df.withColumn("rn", row_number().over(window)) \
                          .filter("rn = 1") \
                          .drop("rn")

    return dedup_df

### Merge Logic

In [0]:
def merge_to_silver(df, table, primary_key):

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.silver_23283.{table}
    USING DELTA
    AS SELECT * FROM bronze_23283.{table} WHERE 1=0
    """)

    df.createOrReplaceTempView("updates")

    spark.sql(f"""
        MERGE INTO {catalog_name}.silver_23283.{table} AS target
        USING updates AS source
        ON target.{primary_key} = source.{primary_key}

        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

Pipeline Execurion

In [0]:
tables = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id"
}

for table, pk in tables.items():
    df = process_table(table, pk)
    merge_to_silver(df, table, pk)

    spark.sql(f"""
        INSERT INTO {catalog_name}.metadata_23283.pipeline_control
        VALUES ('{table}', current_timestamp())
    """)

### Join

In [0]:
orders = spark.table("silver_23283.orders")
customers = spark.table("silver_23283.customers")

df = orders.join(customers, "customer_id", "left")